# noise-batch-from-latent — worked example 2: Reproducible noise batch using a torch.Generator

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `noise-batch-from-latent`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

When debugging or running reproducibility checks, you need the same noise batch every time. Passing a seeded `torch.Generator` object to `t.randn` achieves this without fixing the global seed — which would affect all other random calls in your program. The generator is local state, so two calls with two different generators can produce different batches even within the same iteration.

## Worked solution

**Step 1 — Create and seed the generator.**
We instantiate `t.Generator()` and call `.manual_seed(seed)` on it. This is completely independent of the global `t.manual_seed` state, so other random draws in the program are unaffected.

**Step 2 — Pass the generator to `t.randn`.**
The `generator=g` keyword argument routes all random draws through our seeded generator. The result is fully reproducible: the same seed always yields the same tensor.

**Step 3 — Call twice and compare.**
To verify reproducibility, we rebuild the generator with the same seed and draw again. The two tensors must be element-wise identical. We also verify they are NOT identical to a draw made with a different seed.

**Step 4 — Check shape and dtype.**
Shape is `(batch_size, latent_dim)`. Default dtype is `float32`, which is correct for GAN inputs.

In [ ]:
import torch as t

def seeded_noise(batch_size: int, latent_dim: int, seed: int) -> t.Tensor:
    """Return reproducible (B, L) noise using a local generator."""
    g = t.Generator()
    g.manual_seed(seed)
    return t.randn(batch_size, latent_dim, generator=g)

# --- exercise it ---
B, L = 6, 64

noise_a = seeded_noise(B, L, seed=7)
noise_b = seeded_noise(B, L, seed=7)   # same seed -> identical
noise_c = seeded_noise(B, L, seed=99)  # different seed -> different

print(f'shape : {noise_a.shape}')                          # [6, 64]
print(f'a == b: {t.allclose(noise_a, noise_b)}')           # True
print(f'a == c: {t.allclose(noise_a, noise_c)}')           # False
assert noise_a.shape == (B, L)
assert noise_a.dtype == t.float32
assert t.allclose(noise_a, noise_b)
assert not t.allclose(noise_a, noise_c)